
# iFinD **HTTP API** 一次性下载：融资融券 + 陆股通（北向）

> 用 HTTP API(`requests`)，**不用 iFinDPy SDK**。
> ⚠️ **一次性历史拉取，绝不接进 CSMAR 日更**(日更只走 CSMAR)；需更新手动重跑。
> ⚠️ **北向 2024-08 起监管停盘中实时 + 减个股披露** → 陆股通只在 ~2024.8 前完整(回测可用)。融资融券无此问题。
> ⚠️ **指标 mnemonic 需在 iFinD「指标提取」核对**(我填了常见 ths_* 写法，版本可能不同；错误码 → 按提示改)。
> 产出：`<CSMAR>/raw/IFIND_Margin.parquet` / `IFIND_Northbound.parquet`(long: stkcd/trddt/字段)。


## 1. 取 access_token（用 refresh_token）

In [1]:

import os, json, time
import requests, pandas as pd

DATA_ROOT = os.environ.get("CSMAR_DATA_ROOT", "/Users/shenboheng/CSMAR")
RAW = os.path.join(DATA_ROOT, "raw")
BASE = "https://quantapi.51ifind.com/api/v1"

# refresh_token：优先环境变量；否则从你 01_ETF 的 .ifind_access_token.json 读('refresh' 字段)
REFRESH = os.environ.get("IFIND_REFRESH_TOKEN", "")
if not REFRESH:
    for p in [os.path.expanduser("~/Documents/ClaudeCode/How_to_be_a_quant/Strategy/01_ETF_style_rotation/data/raw/.ifind_access_token.json")]:
        if os.path.exists(p):
            REFRESH = json.load(open(p)).get("refresh", ""); break
assert REFRESH, "未找到 refresh_token：设环境变量 IFIND_REFRESH_TOKEN 或确认 .ifind_access_token.json"

r = requests.post(f"{BASE}/get_access_token",
                  headers={"Content-Type": "application/json", "refresh_token": REFRESH})
ACCESS = json.loads(r.content)["data"]["access_token"]
HEADERS = {"Content-Type": "application/json", "access_token": ACCESS}
print("access_token 取得:", ACCESS[:12], "...")


access_token 取得: c6ab2b39e835 ...


## 2. 股票池 → iFinD 代码

In [2]:

dim = pd.read_parquet(os.path.join(DATA_ROOT, "mart", "dim_stock.parquet"))
def to_ifind(c):
    c = str(c).zfill(6)
    return c + (".SH" if c[0] == "6" else ".BJ" if c[0] in "489" else ".SZ")
codes = [to_ifind(c) for c in dim["stkcd"].unique()]
print("股票数:", len(codes), codes[:3])
START, END = "2015-01-01", "2025-12-31"


股票数: 5896 ['000001.SZ', '000002.SZ', '000004.SZ']


## 3. 指标配置（⚠️ 在 iFinD「指标提取」核对 ths_* 名）

In [3]:

MARGIN = {   # 融资融券（个股日频）
    "ths_financing_balance_stock":          "融资余额",
    "ths_securities_lending_balance_stock": "融券余额",
    "ths_financing_buy_stock":              "融资买入额",
    "ths_margin_trading_balance_stock":     "融资融券余额",
}
NORTH = {    # 陆股通（北向）个股持股
    "ths_north_bound_hold_share_stock": "陆股通持股数量",
    "ths_north_bound_hold_mv_stock":    "陆股通持股市值",
    "ths_north_bound_hold_ratio_stock": "陆股通持股占比",
}
FUNC = {"Days": "Tradedays", "Fill": "Blank", "Interval": "D"}


## 4. date_sequence 拉取 + 解析（分块）

In [4]:

def fetch(fields, chunk=30):
    inds = [{"indicator": k, "indiparams": [""]} for k in fields]
    frames = []
    for i in range(0, len(codes), chunk):
        body = {"codes": ",".join(codes[i:i+chunk]), "indipara": inds,
                "startdate": START, "enddate": END, "functionpara": FUNC}
        try:
            resp = requests.post(f"{BASE}/date_sequence", headers=HEADERS, json=body)
            j = json.loads(resp.content)
            if j.get("errorcode", -1) != 0:
                print("  errcode", j.get("errorcode"), j.get("errmsg", "")[:50]); continue
            for tb in j.get("tables", []):
                t = tb.get("table", {})
                df = pd.DataFrame(t)
                df["stkcd"] = str(tb.get("thscode", ""))[:6]
                df["trddt"] = tb.get("time", [])
                frames.append(df)
        except Exception as e:
            print("  chunk", i, "err", str(e)[:60])
        if i % 300 == 0: print(f"  {i}/{len(codes)} ...", flush=True)
        time.sleep(0.25)
    out = pd.concat(frames, ignore_index=True)
    out["trddt"] = pd.to_datetime(out["trddt"])
    return out.rename(columns={k: v for k, v in {**MARGIN, **NORTH}.items()})


In [5]:

margin = fetch(MARGIN); margin.to_parquet(os.path.join(RAW, "IFIND_Margin.parquet"))
print("融资融券:", margin.shape, "→ raw/IFIND_Margin.parquet")


  0/5896 ...
  300/5896 ...


KeyboardInterrupt: 

In [ ]:

north = fetch(NORTH); north.to_parquet(os.path.join(RAW, "IFIND_Northbound.parquet"))
print("陆股通:", north.shape, "→ raw/IFIND_Northbound.parquet")



## 5. 更新 / 入因子
- **更新**：手动重跑本 notebook（改 END），**不要**进 `update_daily.py`（日更只走 CSMAR）。
- 数据为 `stkcd/trddt/字段` long 格式，落 `raw/IFIND_*.parquet`，可直接被 `quantlib` 读取。
- 跑通后告诉我，我在 `quantlib/altdata.py` 加 **两融情绪**(融资余额变化/融资买入占比/两融余额占流通市值) 和 **北向持股变动** 因子 → 喂因子择时(情绪驱动)。
- 若某 `ths_*` 报错：在 iFinD 终端「数据接口→指标提取」搜对应中文确认正确字段名后替换。
